# Barter Python: Instruments & Market Data

This notebook covers the data layer of the Barter Python bindings:
instruments, indexed lookups, single-exchange streaming, multi-exchange
composition, and the WebSocket integration surface.

## Topics Covered
- Defining spot and perpetual instruments
- `IndexedInstruments` for O(1) indexed lookups
- JSON serialisation of instruments
- Live public-trade streaming via `build_market_stream()`
- Multi-exchange unified streams
- WebSocket integration surface and current Python boundary


## Prerequisites

These notebooks assume `barter` is installed into the active IPython kernel, for example:

```bash
cd /mnt/developer/git/aecs4u.it/quant/barter-rs/barter-python
uv sync --dev
source .venv/bin/activate
python -m pip install -e .
```

If you register a dedicated kernel, select that kernel before running the notebook.


In [ ]:
from pathlib import Path
import sys

def find_repo_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        python_dir = candidate / "barter-python" / "python" / "barter"
        if (python_dir / "__init__.py").exists():
            return candidate
    raise RuntimeError("Could not locate the barter-rs repository root from the current working directory.")

REPO_ROOT = find_repo_root()
PYTHON_SOURCE = REPO_ROOT / "barter-python" / "python"
if str(PYTHON_SOURCE) not in sys.path:
    sys.path.insert(0, str(PYTHON_SOURCE))

print(f"Using barter from {PYTHON_SOURCE}")


In [ ]:
import json
from collections import Counter

from barter import (
    ExchangeId,
    IndexedInstruments,
    Instrument,
    Subscription,
    build_market_stream,
)


---
## Part 1: Instruments & Index Management


## 1. Define Instruments

Create spot and perpetual instruments from Python while still relying on the same Rust-backed indexed model.

In [3]:
btc_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "btc_usdt", "BTCUSDT", "btc", "usdt")
eth_spot = Instrument.spot(ExchangeId.BINANCE_SPOT, "eth_usdt", "ETHUSDT", "eth", "usdt")
btc_perp = Instrument.new_instrument(
    ExchangeId.BINANCE_FUTURES_USD,
    "btc_usdt_perp",
    "BTCUSDT",
    "btc",
    "usdt",
    kind="perpetual",
)

for instrument in [btc_spot, eth_spot, btc_perp]:
    print(instrument)

binance_spot:BTCUSDT (spot)
binance_spot:ETHUSDT (spot)
binance_futures_usd:BTCUSDT (perpetual)


## 2. Build Indexed Lookups

`IndexedInstruments` materializes compact integer indices for exchanges, assets, and instruments.

In [4]:
indexed = IndexedInstruments([btc_spot, eth_spot, btc_perp])

print(indexed)
print("num_exchanges", indexed.num_exchanges)
print("num_assets", indexed.num_assets)
print("exchanges", indexed.exchanges())
print("instruments", indexed.instruments())
print("binance_spot index", indexed.find_exchange_index(ExchangeId.BINANCE_SPOT))

IndexedInstruments(exchanges=2, assets=5, instruments=3)
num_exchanges 2
num_assets 5
exchanges [(0, 'binance_futures_usd'), (1, 'binance_spot')]
instruments [(0, 'btc_usdt_perp', 'BTCUSDT', 'binance_futures_usd', 'perpetual'), (1, 'btc_usdt', 'BTCUSDT', 'binance_spot', 'spot'), (2, 'eth_usdt', 'ETHUSDT', 'binance_spot', 'spot')]
binance_spot index 1


## 3. JSON Serialization

The instrument objects round-trip through JSON cleanly, which is useful when feeding `SystemConfig` or notebook fixtures.

In [5]:
payload = btc_perp.to_json()
round_trip = Instrument.from_json(payload)

print(payload)
print(round_trip)
print(json.loads(payload))

{
  "exchange": "binance_futures_usd",
  "name_internal": "btc_usdt_perp",
  "name_exchange": "BTCUSDT",
  "underlying": {
    "base": {
      "name_internal": "btc",
      "name_exchange": "btc"
    },
    "quote": {
      "name_internal": "usdt",
      "name_exchange": "usdt"
    }
  },
  "quote": "UnderlyingQuote",
  "kind": {
    "perpetual": {
      "contract_size": "1",
      "settlement_asset": {
        "name_internal": "usdt",
        "name_exchange": "usdt"
      }
    }
  },
  "spec": null
}
binance_futures_usd:BTCUSDT (perpetual)
{'exchange': 'binance_futures_usd', 'name_internal': 'btc_usdt_perp', 'name_exchange': 'BTCUSDT', 'underlying': {'base': {'name_internal': 'btc', 'name_exchange': 'btc'}, 'quote': {'name_internal': 'usdt', 'name_exchange': 'usdt'}}, 'quote': 'UnderlyingQuote', 'kind': {'perpetual': {'contract_size': '1', 'settlement_asset': {'name_internal': 'usdt', 'name_exchange': 'usdt'}}}, 'spec': None}


---
## Part 2: Market Data Streaming


The current Python binding exposes live public-trade streaming via
`Subscription` and `build_market_stream()`. L1 and L2 order-book helpers
from the Rust side are not exposed yet.


## 1. Stream Public Trades

Use top-level `await` in IPython to open the stream and collect a few trade events.

In [3]:
def event_to_dict(event):
    trade = event.trade
    return {
        "time_exchange": event.time_exchange,
        "exchange": str(event.exchange),
        "instrument": event.instrument,
        "kind": event.kind,
        "trade_id": None if trade is None else trade.id,
        "price": None if trade is None else trade.price,
        "amount": None if trade is None else trade.amount,
        "side": None if trade is None else str(trade.side),
    }


async def collect_events(subscriptions, limit=5):
    stream = build_market_stream(subscriptions)
    events = []
    async for event in stream:
        events.append(event_to_dict(event))
        if len(events) >= limit:
            break
    return events


subscriptions = [Subscription("binance_spot", "btc", "usdt")]
trade_events = await collect_events(subscriptions, limit=5)
trade_events

[{'time_exchange': '2026-04-10T10:49:04.798+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765813',
  'price': 71821.18,
  'amount': 0.00036,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:05.260+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765814',
  'price': 71821.18,
  'amount': 0.00152,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:05.298+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765815',
  'price': 71821.18,
  'amount': 0.00055,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:05.347+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765816',
  'price': 71821.18,
  'amount': 0.00678,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:06.458+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_

## 2. Multiple Instruments on One Stream

The Python binding normalizes all returned items into the same `MarketEvent` shape.

In [4]:
subscriptions = [
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("binance_spot", "eth", "usdt"),
]

more_events = await collect_events(subscriptions, limit=8)
more_events

[{'time_exchange': '2026-04-10T10:49:07.815+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3920539299',
  'price': 2194.52,
  'amount': 0.4559,
  'side': 'buy'},
 {'time_exchange': '2026-04-10T10:49:08.196+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765903',
  'price': 71819.85,
  'amount': 0.00132,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:08.202+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade_id': '6204765904',
  'price': 71819.85,
  'amount': 0.00132,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:08.292+00:00',
  'exchange': 'binance_spot',
  'instrument': 'eth_usdt_spot',
  'kind': 'trade',
  'trade_id': '3920539300',
  'price': 2194.51,
  'amount': 0.22,
  'side': 'sell'},
 {'time_exchange': '2026-04-10T10:49:08.390+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',


## Binding Notes

- Supported today: public trades via `build_market_stream()`
- Not exposed yet: the Rust notebook's dedicated L1/L2 order-book managers
- The event objects are still produced by the Rust runtime under the hood; the notebook just consumes them from Python


---
## Part 3: Multi-Exchange Streams


## 1. Build a Unified Stream

The Python API accepts multiple subscriptions and yields a unified async stream of `MarketEvent` values.

In [3]:
async def collect_events(subscriptions, limit=12):
    stream = build_market_stream(subscriptions)
    rows = []
    async for event in stream:
        rows.append({
            "exchange": str(event.exchange),
            "instrument": event.instrument,
            "kind": event.kind,
            "price": None if event.trade is None else event.trade.price,
            "amount": None if event.trade is None else event.trade.amount,
        })
        if len(rows) >= limit:
            break
    return rows


subscriptions = [
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("binance_futures_usd", "btc", "usdt", instrument_kind="perpetual"),
    Subscription("coinbase", "eth", "usd"),
]

events = await collect_events(subscriptions)
events

[{'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'price': 2195.29,
  'amount': 0.01913742},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 71814.92,
  'amount': 0.00221},
 {'exchange': 'binance_futures_usd',
  'instrument': 'btc_usdt_perpetual',
  'kind': 'trade',
  'price': 71779.0,
  'amount': 0.032},
 {'exchange': 'binance_futures_usd',
  'instrument': 'btc_usdt_perpetual',
  'kind': 'trade',
  'price': 71778.9,
  'amount': 0.061},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 71814.93,
  'amount': 0.00018},
 {'exchange': 'binance_futures_usd',
  'instrument': 'btc_usdt_perpetual',
  'kind': 'trade',
  'price': 71779.0,
  'amount': 0.005},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 71814.93,
  'amount': 0.00159},
 {'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'price': 71814.93

## 2. Aggregate the Joined Stream

Because all items share one shape, plain Python grouping works well once the events are in memory.

In [4]:
by_exchange = Counter(row["exchange"] for row in events)
by_instrument = Counter(row["instrument"] for row in events)

by_exchange, by_instrument

(Counter({'binance_spot': 5, 'binance_futures_usd': 4, 'coinbase': 3}),
 Counter({'btc_usdt_spot': 5, 'btc_usdt_perpetual': 4, 'eth_usd_spot': 3}))

## 3. Python Binding Differences From Rust

The Rust `Streams::builder_multi()` API exposes more typed composition options. The current Python layer keeps the surface smaller: build a list of `Subscription` specs and consume the unified stream directly.

---
## Part 4: WebSocket Integration Surface


## Current Python Boundary

The Rust `barter-integration` crate exposes low-level transformer, validator, and transport traits. Those hooks are not part of the public Python package yet. In Python, the supported integration surface is the normalized async stream returned by `build_market_stream()`.

In [3]:
def snapshot_event(event):
    trade = event.trade
    return {
        "time_exchange": event.time_exchange,
        "time_received": event.time_received,
        "exchange": str(event.exchange),
        "instrument": event.instrument,
        "kind": event.kind,
        "trade": None if trade is None else {
            "id": trade.id,
            "price": trade.price,
            "amount": trade.amount,
            "side": str(trade.side),
        },
    }


async def capture(subscription, limit=3):
    stream = build_market_stream([subscription])
    rows = []
    async for event in stream:
        rows.append(snapshot_event(event))
        if len(rows) >= limit:
            break
    return rows


rows = await capture(Subscription("binance_spot", "btc", "usdt"), limit=3)
rows

[{'time_exchange': '2026-04-10T10:48:44.876+00:00',
  'time_received': '2026-04-10T10:48:45.034048456+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6204765701',
   'price': 71823.88,
   'amount': 0.00054,
   'side': 'sell'}},
 {'time_exchange': '2026-04-10T10:48:45.027+00:00',
  'time_received': '2026-04-10T10:48:45.152044493+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6204765702',
   'price': 71823.88,
   'amount': 7e-05,
   'side': 'sell'}},
 {'time_exchange': '2026-04-10T10:48:46.305+00:00',
  'time_received': '2026-04-10T10:48:46.424789565+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6204765703',
   'price': 71823.88,
   'amount': 0.0003,
   'side': 'sell'}}]

## Event-Normalization Pattern

The practical Python pattern is: open a stream, normalize each `MarketEvent` into plain Python data, and then plug that into your notebook analysis, async app, or downstream pipeline.

In [4]:
async def capture_many(subscriptions, limit=6):
    stream = build_market_stream(subscriptions)
    rows = []
    async for event in stream:
        rows.append(snapshot_event(event))
        if len(rows) >= limit:
            break
    return rows


rows = await capture_many([
    Subscription("binance_spot", "btc", "usdt"),
    Subscription("coinbase", "eth", "usd"),
], limit=6)
rows

[{'time_exchange': '2026-04-10T10:48:44.146703+00:00',
  'time_received': '2026-04-10T10:48:47.189580477+00:00',
  'exchange': 'coinbase',
  'instrument': 'eth_usd_spot',
  'kind': 'trade',
  'trade': {'id': '796126283',
   'price': 2195.47,
   'amount': 0.00450017,
   'side': 'sell'}},
 {'time_exchange': '2026-04-10T10:48:48.054+00:00',
  'time_received': '2026-04-10T10:48:48.175359462+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6204765705',
   'price': 71823.88,
   'amount': 0.00243,
   'side': 'sell'}},
 {'time_exchange': '2026-04-10T10:48:48.984+00:00',
  'time_received': '2026-04-10T10:48:49.130033462+00:00',
  'exchange': 'binance_spot',
  'instrument': 'btc_usdt_spot',
  'kind': 'trade',
  'trade': {'id': '6204765706',
   'price': 71823.89,
   'amount': 0.002,
   'side': 'buy'}},
 {'time_exchange': '2026-04-10T10:48:48.984+00:00',
  'time_received': '2026-04-10T10:48:49.130046908+00:00',
  'exchange': 'binance_spot

## What Rust Still Has That Python Does Not

- Custom transformer and validator traits
- Direct access to protocol-level websocket plumbing
- The lower-level extension points used to build new exchange integrations

Those remain Rust-only for now, so this notebook stays at the Python-supported integration layer.